# Clustering
This notebook allows us to project embeddings into a lower dimensional space and then cluster them. Based on our earlier result, we know that around 9 dimensions in sufficient to retrain descrimination between different classes. We will use PCA, is the fastest and has been shown to be consistently the best.

We are going to try clustering with:
- KMeans
- DBScan
- Spectral Clustering

In [1]:
from hproj.util.seeds import set_seeds


set_seeds()

In [2]:
from dataclasses import dataclass

from matplotlib import projections

from hproj.data.feature_space import FeatureSpace
from hproj.data.paths import Paths
from hproj.projectors.pca import PCAProjector


encoders = [
    'uni',
    'uni2',
    'hibou',
    'dinov2'
]

datasets = [
    'kather100k',
    'spider-colorectal',
    'spider-breast',
    'spider-skin',
    'spider-thorax'
]

n_dims = 9

paths = Paths.from_env()

data = {}

@dataclass
class DatasetRepresentation:
    train: FeatureSpace
    test: FeatureSpace

@dataclass
class DatasetRepresentationSet:
    # these are the orginal embeddings, before projection
    embeddings: DatasetRepresentation

    # these are the projected embeddings where the key is the number of components and the value is the projected feature space
    projections: dict[int, DatasetRepresentation]    


for encoder in encoders:
    for dataset in datasets:
        # load the embeddings and labels
        embeddings_path = paths.embedding(dataset, encoder)
        train, test = embeddings_path.load_splits()

        # compute the PCA projection
        projector = PCAProjector(n_components=n_dims, seed=42)
        projector.fit(train)
        train_proj = projector.transform(train)
        test_proj = projector.transform(test)

        # store the projected data
        data[(encoder, dataset)] = DatasetRepresentationSet(
            embeddings=DatasetRepresentation(train=train, test=test),
            projections={n_dims: DatasetRepresentation(train=train_proj, test=test_proj)}
        )

# KMeans
The method for kmeans is:
Using the training set:
- scale the features (kmeans assumes roughly spherical clusters) - fit the scalar on the training set
- search over several values of k (we know there are between 9 and 24 classes so k = 2..32 seems smart)
- plot the elbow curve and silhoette scores and use this to select a best k
- take the clusterer that was fit to the training set with k and use it to predict cluster labels on test set
- on the test set measure silhoette, adjusted_rand_score, and adjusted_mutual_info_score

In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cuml.cluster import KMeans
from cuml.preprocessing import StandardScaler
from cuml.metrics.cluster import silhouette_score
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


def search_k(X_train_scaled, k_values):
    """Fit KMeans for each k, returning inertias and silhouette scores."""
    inertias  = []
    sil_train = []
    for k in k_values:
        km     = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_train_scaled)
        inertias.append(float(km.inertia_))
        sil_train.append(float(silhouette_score(X_train_scaled, labels, metric='cosine')))
    return inertias, np.array(sil_train)


def plot_elbow_and_silhouette(k_values, inertias, sil_arr, best_k, title):
    """Plot the elbow curve and silhouette scores, marking the best k."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=13)

    ax1.plot(k_values, inertias, 'b-o', ms=4)
    ax1.axvline(best_k, color='r', linestyle='--', label=f'best k={best_k}')
    ax1.set_xlabel('k')
    ax1.set_ylabel('Inertia')
    ax1.set_title('Elbow Curve')
    ax1.legend()

    ax2.plot(k_values, sil_arr, 'g-o', ms=4)
    ax2.axvline(best_k, color='r', linestyle='--', label=f'best k={best_k}')
    ax2.set_xlabel('k')
    ax2.set_ylabel('Silhouette Score')
    ax2.set_title('Silhouette Scores (train)')
    ax2.legend()

    plt.tight_layout()
    plt.show()


def evaluate_kmeans(X_train_scaled, X_test_scaled, y_test, best_k):
    """Refit KMeans with best_k and evaluate on the test set."""
    best_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    best_km.fit(X_train_scaled)

    test_cluster_labels = best_km.predict(X_test_scaled)

    test_sil = float(silhouette_score(X_test_scaled, test_cluster_labels, metric='cosine'))
    ari = adjusted_rand_score(y_test.get(), test_cluster_labels.get())
    ami = adjusted_mutual_info_score(y_test.get(), test_cluster_labels.get())

    return {'best_k': best_k, 'silhouette': test_sil, 'ari': ari, 'ami': ami}


def kmeans_cluster_and_evaluate(
    data: dict[tuple[str, str], DatasetRepresentationSet],
    k_values: list[int],
    plot: bool = False,
) -> pd.DataFrame:
    """Evaluate KMeans on full embeddings and all PCA projections in each DatasetRepresentationSet.

    Returns a DataFrame with columns: encoder, dataset, representation, best_k, silhouette, ari, ami.
    """
    rows = []

    for (encoder, dataset), rep_set in data.items():
        for rep_label, rep in [('full', rep_set.embeddings), *rep_set.projections.items()]:
            X_train = rep.train.features
            X_test  = rep.test.features
            y_test  = rep.test.labels

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled  = scaler.transform(X_test)

            inertias, sil_arr = search_k(X_train_scaled, k_values)
            best_k = k_values[int(np.argmax(sil_arr))]

            if plot:
                plot_elbow_and_silhouette(
                    k_values, inertias, sil_arr, best_k,
                    f'{encoder} / {dataset} ({rep_label} dims)'
                )

            result = evaluate_kmeans(X_train_scaled, X_test_scaled, y_test, best_k)
            rows.append({
                'encoder':        encoder,
                'dataset':        dataset,
                'representation': rep_label,
                **result,
            })

            print(
                f"{encoder:8s} / {dataset:25s} | {str(rep_label):6s} | "
                f"k={result['best_k']:2d} | "
                f"sil={result['silhouette']:.4f} | "
                f"ari={result['ari']:.4f} | "
                f"ami={result['ami']:.4f}"
            )

    return pd.DataFrame(rows)


k_values = list(range(2, 33))

kmeans_results = kmeans_cluster_and_evaluate(data, k_values, plot=False)
kmeans_results


# Density Based Methods

In [3]:

import itertools

import numpy as np
import pandas as pd
from cuml.cluster.hdbscan import HDBSCAN, approximate_predict
from cuml.metrics.cluster import silhouette_score
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.preprocessing import normalize


def preprocess_for_hdbscan(X_train, X_test):
    """L2-normalise embeddings so Euclidean distance proxies cosine similarity."""
    X_train_proc = normalize(X_train.get() if hasattr(X_train, 'get') else X_train, norm='l2')
    X_test_proc  = normalize(X_test.get()  if hasattr(X_test,  'get') else X_test,  norm='l2')
    return X_train_proc, X_test_proc


def search_hdbscan(X_train_proc, param_grid):
    """Fit HDBSCAN for every (min_cluster_size, min_samples) pair.

    Returns a DataFrame with one row per configuration.
    """
    rows = []
    for min_cluster_size, min_samples in param_grid:
        model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
        )
        train_labels = model.fit_predict(X_train_proc)

        train_labels_np = train_labels.get() if hasattr(train_labels, 'get') else np.asarray(train_labels)
        mask = train_labels_np != -1
        n_clusters   = int(np.unique(train_labels_np[mask]).shape[0]) if mask.any() else 0
        noise_frac   = float((~mask).mean())

        if n_clusters >= 2:
            sil = float(silhouette_score(X_train_proc[mask], train_labels_np[mask], metric='euclidean'))
        else:
            sil = float('nan')

        persistence = model.cluster_persistence_
        if persistence is not None and len(persistence) > 0:
            mean_persistence = float(np.mean(
                persistence.get() if hasattr(persistence, 'get') else persistence
            ))
        else:
            mean_persistence = float('nan')

        rows.append({
            'min_cluster_size':   min_cluster_size,
            'min_samples':        min_samples,
            'train_silhouette':   sil,
            'n_clusters_train':   n_clusters,
            'noise_frac_train':   noise_frac,
            'mean_persistence':   mean_persistence,
        })

    return pd.DataFrame(rows)


def select_best_hdbscan(search_df, expected_range=(9, 24)):
    """Apply constrained selection: filter invalid runs, rank by quality."""
    valid = search_df[
        (search_df['n_clusters_train'] >= 2) &
        (search_df['noise_frac_train'] <= 0.50) &
        (search_df['n_clusters_train'] <= 40)
    ].copy()

    if valid.empty:
        # Fall back to least-bad: most clusters, lowest noise
        valid = search_df.sort_values(
            ['n_clusters_train', 'noise_frac_train'], ascending=[False, True]
        ).head(1)

    lo, hi = expected_range
    valid['cluster_dist'] = (valid['n_clusters_train'] - (lo + hi) / 2).abs()

    valid = valid.sort_values(
        ['train_silhouette', 'cluster_dist', 'noise_frac_train', 'mean_persistence'],
        ascending=[False, True, True, False],
        na_position='last',
    )

    best = valid.iloc[0]
    return {
        'min_cluster_size': int(best['min_cluster_size']),
        'min_samples':      None if pd.isna(best['min_samples']) else int(best['min_samples']),
    }


def evaluate_hdbscan(X_train_proc, X_test_proc, y_test, best_params):
    """Fit HDBSCAN with best params on train, predict test via approximate_predict."""
    model = HDBSCAN(
        min_cluster_size=best_params['min_cluster_size'],
        min_samples=best_params['min_samples'],
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )
    train_labels = model.fit_predict(X_train_proc)
    train_labels_np = train_labels.get() if hasattr(train_labels, 'get') else np.asarray(train_labels)

    train_mask      = train_labels_np != -1
    n_clusters_train = int(np.unique(train_labels_np[train_mask]).shape[0]) if train_mask.any() else 0
    noise_frac_train = float((~train_mask).mean())

    test_labels, _strengths = approximate_predict(model, X_test_proc)
    test_labels_np = test_labels.get() if hasattr(test_labels, 'get') else np.asarray(test_labels)

    test_mask       = test_labels_np != -1
    n_clusters_test = int(np.unique(test_labels_np[test_mask]).shape[0]) if test_mask.any() else 0
    noise_frac_test = float((~test_mask).mean())

    y_test_np = y_test.get() if hasattr(y_test, 'get') else np.asarray(y_test)
    ari = adjusted_rand_score(y_test_np, test_labels_np)
    ami = adjusted_mutual_info_score(y_test_np, test_labels_np)

    if test_mask.any() and n_clusters_test >= 2:
        test_sil = float(silhouette_score(X_test_proc[test_mask], test_labels_np[test_mask], metric='euclidean'))
    else:
        test_sil = float('nan')

    persistence = model.cluster_persistence_
    if persistence is not None and len(persistence) > 0:
        mean_persistence = float(np.mean(
            persistence.get() if hasattr(persistence, 'get') else persistence
        ))
    else:
        mean_persistence = float('nan')

    return {
        'best_min_cluster_size': best_params['min_cluster_size'],
        'best_min_samples':      best_params['min_samples'],
        'n_clusters_train':      n_clusters_train,
        'noise_frac_train':      noise_frac_train,
        'n_clusters_test':       n_clusters_test,
        'noise_frac_test':       noise_frac_test,
        'silhouette':            test_sil,
        'ari':                   ari,
        'ami':                   ami,
        'mean_persistence_train': mean_persistence,
    }


def hdbscan_cluster_and_evaluate(
    data: dict,
    min_cluster_sizes: list[int] | None = None,
    min_samples_values: list[int | None] | None = None,
    expected_cluster_range: tuple[int, int] = (9, 24),
) -> pd.DataFrame:
    """Evaluate HDBSCAN on full embeddings and all PCA projections.

    Returns a DataFrame with columns mirroring the KMeans results plus
    HDBSCAN-specific columns (noise fractions, cluster counts, persistence).
    """
    if min_cluster_sizes is None:
        min_cluster_sizes = [5, 10, 15, 20, 30, 40, 60]
    if min_samples_values is None:
        min_samples_values = [None, 5, 10, 15, 20]

    param_grid = list(itertools.product(min_cluster_sizes, min_samples_values))

    rows = []

    for (encoder, dataset), rep_set in data.items():
        for rep_label, rep in [('full', rep_set.embeddings), *rep_set.projections.items()]:
            X_train = rep.train.features
            X_test  = rep.test.features
            y_test  = rep.test.labels

            X_train_proc, X_test_proc = preprocess_for_hdbscan(X_train, X_test)

            search_df = search_hdbscan(X_train_proc, param_grid)
            best_params = select_best_hdbscan(search_df, expected_range=expected_cluster_range)

            result = evaluate_hdbscan(X_train_proc, X_test_proc, y_test, best_params)
            rows.append({
                'encoder':        encoder,
                'dataset':        dataset,
                'representation': rep_label,
                **result,
            })

            print(
                f"{encoder:8s} / {dataset:25s} | {str(rep_label):6s} | "
                f"mcs={result['best_min_cluster_size']:3d} ms={str(result['best_min_samples']):4s} | "
                f"k_tr={result['n_clusters_train']:2d} nf_tr={result['noise_frac_train']:.2f} | "
                f"k_te={result['n_clusters_test']:2d} nf_te={result['noise_frac_test']:.2f} | "
                f"sil={result['silhouette']:.4f} | "
                f"ari={result['ari']:.4f} | "
                f"ami={result['ami']:.4f}"
            )

    return pd.DataFrame(rows)


hdbscan_results = hdbscan_cluster_and_evaluate(data)
hdbscan_results


uni      / kather100k                | full   | mcs= 60 ms=None | k_tr=38 nf_tr=0.40 | k_te= 7 nf_te=0.88 | sil=0.3990 | ari=0.0401 | ami=0.2279
uni      / kather100k                | 9      | mcs= 60 ms=None | k_tr=16 nf_tr=0.15 | k_te=11 nf_te=0.61 | sil=0.8051 | ari=0.2124 | ami=0.5444
uni      / spider-colorectal         | full   | mcs= 40 ms=None | k_tr= 2 nf_tr=0.31 | k_te= 2 nf_te=0.40 | sil=0.0901 | ari=0.0576 | ami=0.1872
uni      / spider-colorectal         | 9      | mcs= 60 ms=15   | k_tr=10 nf_tr=0.47 | k_te=10 nf_te=0.50 | sil=0.6650 | ari=0.1675 | ami=0.5362
uni      / spider-breast             | full   | mcs= 60 ms=None | k_tr=23 nf_tr=0.45 | k_te=13 nf_te=0.63 | sil=0.1704 | ari=0.0575 | ami=0.3102
uni      / spider-breast             | 9      | mcs=  5 ms=None | k_tr=204 nf_tr=0.66 | k_te=47 nf_te=0.84 | sil=0.4211 | ari=0.0214 | ami=0.2552


KeyboardInterrupt: 

# Spectral Density


# Spectral Clustering

The method for spectral clustering is:
Using the training set:
- L2-normalize embeddings (same geometry as HDBSCAN for fair comparison)
- Search over k values (2..32) and two affinity modes: RBF with a data-adaptive γ, and nearest-neighbors
- Select the best (k, affinity, params) by silhouette on the training set
- Predict test labels by fitting a fresh model with the chosen params on train
- Evaluate on test with silhouette, ARI, and AMI


In [ ]:

import numpy as np
import pandas as pd
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, pairwise_distances, silhouette_score as sk_silhouette_score
from sklearn.preprocessing import normalize


def preprocess_for_spectral(X_train, X_test):
    """L2-normalise embeddings for consistent geometry with HDBSCAN."""
    X_train_proc = normalize(X_train.get() if hasattr(X_train, 'get') else X_train, norm='l2')
    X_test_proc  = normalize(X_test.get()  if hasattr(X_test,  'get') else X_test,  norm='l2')
    return X_train_proc, X_test_proc


def compute_rbf_gamma(X):
    """Return gamma = 1 / (2 * median_pairwise_distance^2), adapted to dataset scale."""
    dists = pairwise_distances(X, metric='euclidean')
    upper = dists[np.triu_indices_from(dists, k=1)]
    median_dist = float(np.median(upper))
    if median_dist == 0:
        return 1.0
    return 1.0 / (2.0 * median_dist ** 2)


def search_spectral(X_train_proc, k_values, gamma):
    """Fit SpectralClustering for each (k, affinity) combination.

    Tries RBF affinity with the heuristic gamma and nearest-neighbors affinity.
    Returns a DataFrame with one row per configuration.
    """
    configs = []
    for k in k_values:
        configs.append({'k': k, 'affinity': 'rbf',               'affinity_params': {'gamma': gamma}})
        configs.append({'k': k, 'affinity': 'nearest_neighbors',  'affinity_params': {'n_neighbors': 15}})

    rows = []
    for cfg in configs:
        sc = SpectralClustering(
            n_clusters=cfg['k'],
            affinity=cfg['affinity'],
            assign_labels='kmeans',
            random_state=42,
            n_jobs=-1,
            **cfg['affinity_params'],
        )
        train_labels = sc.fit_predict(X_train_proc)
        n_clusters   = int(np.unique(train_labels).shape[0])

        if n_clusters >= 2:
            sil = float(sk_silhouette_score(X_train_proc, train_labels, metric='euclidean'))
        else:
            sil = float('nan')

        rows.append({
            'k':           cfg['k'],
            'affinity':    cfg['affinity'],
            'gamma':       cfg['affinity_params'].get('gamma', float('nan')),
            'n_neighbors': cfg['affinity_params'].get('n_neighbors', float('nan')),
            'train_silhouette': sil,
        })

    return pd.DataFrame(rows)


def select_best_spectral(search_df):
    """Pick the config with the highest train silhouette."""
    valid = search_df[search_df['train_silhouette'].notna()]
    if valid.empty:
        valid = search_df
    best = valid.loc[valid['train_silhouette'].idxmax()]
    return {
        'k':           int(best['k']),
        'affinity':    best['affinity'],
        'gamma':       best['gamma'],
        'n_neighbors': None if np.isnan(best['n_neighbors']) else int(best['n_neighbors']),
    }


def evaluate_spectral(X_train_proc, X_test_proc, y_test, best_params):
    """Refit SpectralClustering on train with best params; assign test points to
    the nearest cluster centroid (spectral has no native predict)."""
    affinity_kwargs = {}
    if best_params['affinity'] == 'rbf':
        affinity_kwargs['gamma'] = best_params['gamma']
    else:
        affinity_kwargs['n_neighbors'] = best_params['n_neighbors']

    sc = SpectralClustering(
        n_clusters=best_params['k'],
        affinity=best_params['affinity'],
        assign_labels='kmeans',
        random_state=42,
        n_jobs=-1,
        **affinity_kwargs,
    )
    train_labels = sc.fit_predict(X_train_proc)

    # Assign test points via nearest centroid in the original (normalised) space
    unique_labels = np.unique(train_labels)
    centroids = np.vstack([X_train_proc[train_labels == c].mean(axis=0) for c in unique_labels])
    dists_to_centroids = pairwise_distances(X_test_proc, centroids, metric='euclidean')
    test_labels = unique_labels[np.argmin(dists_to_centroids, axis=1)]

    y_test_np = y_test.get() if hasattr(y_test, 'get') else np.asarray(y_test)
    ari = adjusted_rand_score(y_test_np, test_labels)
    ami = adjusted_mutual_info_score(y_test_np, test_labels)

    n_test_clusters = int(np.unique(test_labels).shape[0])
    if n_test_clusters >= 2:
        test_sil = float(sk_silhouette_score(X_test_proc, test_labels, metric='euclidean'))
    else:
        test_sil = float('nan')

    return {
        'best_k':        best_params['k'],
        'best_affinity': best_params['affinity'],
        'silhouette':    test_sil,
        'ari':           ari,
        'ami':           ami,
    }


def spectral_cluster_and_evaluate(
    data: dict,
    k_values: list[int] | None = None,
) -> pd.DataFrame:
    """Evaluate SpectralClustering on full embeddings and all PCA projections.

    Tries both RBF (heuristic gamma) and nearest-neighbors affinities, selects the
    best by train silhouette, and evaluates on the test set.

    Returns a DataFrame with columns matching the KMeans/HDBSCAN results frames.
    """
    if k_values is None:
        k_values = list(range(2, 33))

    rows = []

    for (encoder, dataset), rep_set in data.items():
        for rep_label, rep in [('full', rep_set.embeddings), *rep_set.projections.items()]:
            X_train = rep.train.features
            X_test  = rep.test.features
            y_test  = rep.test.labels

            X_train_proc, X_test_proc = preprocess_for_spectral(X_train, X_test)

            gamma     = compute_rbf_gamma(X_train_proc)
            search_df = search_spectral(X_train_proc, k_values, gamma)
            best_params = select_best_spectral(search_df)

            result = evaluate_spectral(X_train_proc, X_test_proc, y_test, best_params)
            rows.append({
                'encoder':        encoder,
                'dataset':        dataset,
                'representation': rep_label,
                **result,
            })

            print(
                f"{encoder:8s} / {dataset:25s} | {str(rep_label):6s} | "
                f"k={result['best_k']:2d} affinity={result['best_affinity']:17s} | "
                f"sil={result['silhouette']:.4f} | "
                f"ari={result['ari']:.4f} | "
                f"ami={result['ami']:.4f}"
            )

    return pd.DataFrame(rows)


k_values = list(range(2, 33))

spectral_results = spectral_cluster_and_evaluate(data, k_values)
spectral_results
